# Notebook 02 — Data Retrieval
## Class 5 Skill: Use at least two retrieval modes; show completeness; preserve raw inputs

This notebook demonstrates:
- Retrieval Mode 1: HTTP file download (TLC Parquet + Zone CSV)
- Retrieval Mode 2: REST API / JSON (NYC Open Data SODA metadata)
- Retrieval Mode 3: Local file read (Vendor metadata embedded in repo)
- Completeness checks to confirm retrieval succeeded
- Raw file preservation strategy

> FDE mindset: We must be able to answer *how do I know retrieval is complete* before moving on.

In [ ]:
import sys
sys.path.insert(0, '../pipeline')

import pandas as pd
import json
from pathlib import Path
from ingest import ingest_trips, ingest_zones, ingest_vendors, ingest_metadata, verify_retrieval_completeness

print('All ingest functions imported successfully.')

## Retrieval Mode 1 — HTTP Parquet Download

The TLC publishes monthly trip data as Parquet files on a public CloudFront CDN.  
We download the file with `requests` (streaming, 1 MB chunks) and save it to `data/raw/`.  
If the file already exists, the download is skipped — this makes the pipeline idempotent.

In [ ]:
print('Retrieving TLC Yellow Trip data (Parquet) ...')
trips_df = ingest_trips()
print(f'Shape: {trips_df.shape}')
print(f'Columns: {list(trips_df.columns)}')
trips_df.head(3)

## Retrieval Mode 1 (continued) — HTTP CSV Download

The zone lookup table is a separate CSV file on the same CDN.

In [ ]:
print('Retrieving TLC Zone Lookup (CSV) ...')
zones_df = ingest_zones()
print(f'Shape: {zones_df.shape}')
zones_df.head(5)

## Retrieval Mode 2 — REST API / JSON (NYC Open Data SODA)

We call the SODA metadata endpoint for the TLC Yellow Taxi dataset.  
This returns a JSON object containing field-level metadata (names, descriptions, data types).  
We use this to cross-check our column names and expected value ranges.

URL: `https://data.cityofnewyork.us/api/views/biws-g3hs.json`

In [ ]:
print('Fetching field metadata from NYC Open Data SODA API ...')
metadata = ingest_metadata()

if metadata:
    print(f'Metadata keys: {list(metadata.keys())[:10]}')
    # Show column metadata if available
    if 'columns' in metadata:
        cols_meta = pd.DataFrame(metadata['columns'])
        print(f'\nField metadata ({len(cols_meta)} fields):')
        if 'fieldName' in cols_meta.columns and 'dataTypeName' in cols_meta.columns:
            print(cols_meta[['fieldName', 'dataTypeName']].to_string(index=False))
else:
    print('API returned empty response (possible rate limit or connectivity issue).')
    print('Cached metadata file used if available.')

## Retrieval Mode 3 — Local File Read (Embedded Vendor Metadata)

Vendor names are not in the trip Parquet — they are only documented in the TLC data dictionary PDF.  
We embedded this mapping as a small CSV in the repo (`data/raw/vendor_metadata.csv`) and read it with `pd.read_csv()`.

In [ ]:
print('Loading vendor metadata from local CSV ...')
vendors_df = ingest_vendors()
print(vendors_df.to_string(index=False))

## Completeness Checks — How Do We Know Retrieval Is Complete?

Three automated checks run after loading:
1. **Row count plausibility** — January TLC Yellow data should have 2–3 million rows
2. **Schema check** — all expected columns must be present
3. **All-null columns** — any entirely-null column suggests a corrupted download

In [ ]:
print('Running retrieval completeness checks ...')
try:
    verify_retrieval_completeness(trips_df)
    print('All completeness checks PASSED.')
except RuntimeError as e:
    print(f'COMPLETENESS CHECK FAILED: {e}')

## Raw File Preservation Strategy

All raw files are written to `data/raw/` and **never modified after download**.  
The pipeline always reads from `data/raw/` and writes outputs to `data/processed/` and `outputs/`.  
This means we can always re-validate or re-model from the original source without re-downloading.

In [ ]:
from pathlib import Path

raw_dir = Path('../data/raw')
print('Raw files preserved in data/raw/:')
for f in sorted(raw_dir.iterdir()):
    size_mb = f.stat().st_size / 1e6
    print(f'  {f.name:45s}  {size_mb:8.2f} MB')